# 01 — Exploratory Data Analysis
Dataset: APTOS 2019 Blindness Detection
Goals: download, class balance, image quality, sample viz, create train/val/test splits.

## 1. Download the dataset
1. kaggle.com/settings → Create New API Token → downloads kaggle.json
2. Place at ~/.kaggle/kaggle.json (or C:\Users\<you>\.kaggle\kaggle.json on Windows)
3. pip install kaggle (already in requirements.txt)

In [8]:
import subprocess
result = subprocess.run(["pip", "show", "kaggle"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [9]:
import subprocess
subprocess.run(["pip", "install", "kaggle"], check=True)

CompletedProcess(args=['pip', 'install', 'kaggle'], returncode=0)

In [13]:
import os
kaggle_json = os.path.expanduser("~/.kaggle/kaggle.json")
print(os.path.exists(kaggle_json))

False


In [10]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()
api.competition_download_files("aptos2019-blindness-detection", path=str(RAW_DIR))

Authentication required to call the Kaggle API.

First, you will need a Kaggle account. You can sign up at
  https://www.kaggle.com/account/login

Recommended: log in with OAuth via a web-based authorization flow.
No token to manage; credentials are cached locally for you.
    kaggle auth login

If you'd rather not use OAuth, generate an API token at
  https://www.kaggle.com/settings/api  (click "Generate New Token" under "API")
and supply it to the CLI in one of these ways:

  Option A: Environment variable
    export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx  # token copied from the settings UI

  Option B: API token file
    Save the token to ~/.kaggle/access_token


NameError: name 'exit' is not defined

In [11]:
import os
from pathlib import Path

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

zip_path = RAW_DIR / "aptos2019-blindness-detection.zip"

if not zip_path.exists():
    os.system(f"kaggle competitions download -c aptos2019-blindness-detection -p {RAW_DIR}")
else:
    print("Zip already downloaded, skipping.")

In [7]:
import subprocess
result = subprocess.run(["kaggle", "competitions", "download", "-c", "aptos2019-blindness-detection", "-p", str(RAW_DIR)],
                         capture_output=True, text=True)
print(result.returncode)
print(result.stdout)
print(result.stderr)

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [12]:
import zipfile

if not (RAW_DIR / "train.csv").exists():
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(RAW_DIR)
    print("Extracted.")
else:
    print("Already extracted, skipping.")

print(list(RAW_DIR.iterdir())[:10])

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\raw\\aptos2019-blindness-detection.zip'

In [ ]:
import pandas as pd

df = pd.read_csv(RAW_DIR / "train.csv")
print(f"Total labeled images: {len(df)}")
df.head()

In [ ]:
import matplotlib.pyplot as plt

SEVERITY_LABELS = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]

counts = df["diagnosis"].value_counts().sort_index()
counts.index = [SEVERITY_LABELS[i] for i in counts.index]

ax = counts.plot(kind="bar", figsize=(7, 4), color="#1F4E5F")
ax.set_title("Class distribution — APTOS 2019")
ax.set_ylabel("Number of images")
for i, v in enumerate(counts.values):
    ax.text(i, v + 10, str(v), ha="center")
plt.tight_layout()
plt.show()

print(counts)
print()
print("Class imbalance ratio (largest/smallest):", round(counts.max() / counts.min(), 1))

In [ ]:
from PIL import Image
import random

sample_ids = df["id_code"].sample(10, random_state=42).tolist()
sizes = []
for img_id in sample_ids:
    img_path = RAW_DIR / "train_images" / f"{img_id}.png"
    with Image.open(img_path) as img:
        sizes.append(img.size)

print("Sample image sizes (width, height):")
for img_id, size in zip(sample_ids, sizes):
    print(f"  {img_id}: {size}")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for severity_idx in range(5):
    subset = df[df["diagnosis"] == severity_idx]
    sample_id = subset["id_code"].sample(1, random_state=1).iloc[0]
    img_path = RAW_DIR / "train_images" / f"{sample_id}.png"

    img = Image.open(img_path)
    axes[severity_idx].imshow(img)
    axes[severity_idx].set_title(SEVERITY_LABELS[severity_idx])
    axes[severity_idx].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

df_renamed = df.rename(columns={"id_code": "image_id"})

train_df, temp_df = train_test_split(
    df_renamed, test_size=0.30, stratify=df_renamed["diagnosis"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["diagnosis"], random_state=42
)

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
val_df.to_csv(PROCESSED_DIR / "val.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)

print("Saved train.csv / val.csv / test.csv to data/processed/")

In [ ]:
for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    ratios = split_df["diagnosis"].value_counts(normalize=True).sort_index().round(3)
    print(f"{name}: {ratios.to_dict()}")

In [ ]:
import shutil

images_src = RAW_DIR / "train_images"
images_dst = PROCESSED_DIR / "images"

if not images_dst.exists():
    shutil.copytree(images_src, images_dst)
    print(f"Copied images to {images_dst}")
else:
    print("Images already present at", images_dst)